# 03 — Embeddings and text/sequence classification

Real NLP inputs aren't continuous vectors — they're **discrete token IDs**. This notebook introduces `nn.Embedding` (id → vector), and the simplest sequence model: **embed → mean-pool → linear**. It's a "bag of words" — order-independent — and it nails a task that only depends on *which* tokens appear.

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## Data — sequences of tokens, "does the trigger appear?"

Each example is a length-`S` sequence of token IDs from a vocab of size `V`. Label = 1 if a specific *trigger* token appears anywhere in the sequence. This depends only on token *presence*, not order — ideal for a bag-of-words model.

In [ ]:
V, S, TRIGGER = 12, 16, 7
def make_trigger(n):
    X = torch.randint(0, V, (n, S))
    y = (X == TRIGGER).any(dim=1).long()
    return X, y

Xtr, ytr = make_trigger(3000); Xva, yva = make_trigger(1000)
Xtr, ytr, Xva, yva = (t.to(device) for t in (Xtr, ytr, Xva, yva))
print("example sequence:", Xtr[0].tolist(), "-> label", ytr[0].item())
print("class balance (train):", ytr.float().mean().item())

## Model — embed → mean-pool → linear

`nn.Embedding(V, D)` is a lookup table: each token ID maps to a learnable `D`-vector. Averaging over the sequence axis gives one fixed-size vector per example (a "bag of words"), which a linear layer classifies.

In [ ]:
class MeanPoolClassifier(nn.Module):
    def __init__(self, V, D=32, n_classes=2):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.fc = nn.Linear(D, n_classes)
    def forward(self, x):              # x: (B, S) token ids
        h = self.emb(x)                # (B, S, D)
        h = h.mean(dim=1)              # (B, D)  <- pool over the sequence (order-independent)
        return self.fc(h)

model = MeanPoolClassifier(V).to(device)
print("embedding table shape:", model.emb.weight.shape)   # (V, D)

### What `nn.Embedding` is (and how it differs from `nn.Linear`)

`nn.Embedding(V, D)` is a **lookup table** of shape `(V, D)` — one learnable `D`-vector per vocabulary item. Its forward pass takes **integer token IDs** and **returns the corresponding rows** (`weight[idx]`) — a pure index, no matmul.

| | `nn.Embedding(V, D)` | `nn.Linear(in, out)` |
|---|---|---|
| **Input** | integer IDs (discrete tokens) | float vectors (continuous) |
| **Operation** | row **lookup** (`weight[idx]`) | matrix **multiply** `x Wᵀ + b` |
| **Weight shape** | `(V, D)` | `(out, in)` |
| **Purpose** | discrete symbol → vector | transform a vector |

**The connection:** an embedding lookup is exactly a `Linear`-with-no-bias applied to a **one-hot** input — `onehot(i) @ W = W[i]` picks out row `i`. `nn.Embedding` is just the *efficient* version: instead of building a giant `V`-dim one-hot and doing a matmul that's almost all multiply-by-zero, it indexes the row directly. Same result, none of the waste. (Consequence: an `Embedding` only receives gradients for the **rows actually used** in a batch — *sparse* — whereas a `Linear` gets a dense gradient over its whole weight.)

In [ ]:
emb = nn.Embedding(5, 3)                          # V=5 tokens, D=3 lookup table
ids = torch.tensor([0, 2, 2])
print("emb(ids) shape:", tuple(emb(ids).shape))   # (3, 3): each id -> its 3-dim row
print("lookup == weight row:", torch.equal(emb(ids)[0], emb.weight[0]))       # True
onehot = F.one_hot(ids, num_classes=5).float()    # (3, 5)
print("onehot @ W == emb(ids):", torch.allclose(onehot @ emb.weight, emb(ids)))  # True

## Train and evaluate

In [ ]:
def fit(model, Xtr, ytr, Xva, yva, epochs, lr=1e-2, bs=128):
    opt = torch.optim.AdamW(model.parameters(), lr=lr); hist = {"tr": [], "va": []}
    for _ in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for k in range(0, len(Xtr), bs):
            idx = perm[k:k+bs]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            hist["tr"].append((model(Xtr).argmax(1) == ytr).float().mean().item())
            hist["va"].append((model(Xva).argmax(1) == yva).float().mean().item())
    return hist

hist = fit(model, Xtr, ytr, Xva, yva, epochs=30)
print(f"final val accuracy: {hist['va'][-1]:.3f}")
plt.plot(hist["tr"], label="train acc"); plt.plot(hist["va"], label="val acc")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.title("trigger-token task"); plt.show()

Near-perfect — because the task ("does token 7 appear?") depends only on *presence*, and mean-pooling preserves presence.

## The catch — pooling throws away order

`mean(dim=1)` gives the **same** vector for any permutation of a sequence. So this model is blind to word order. It can tell "the token 7 is here," never "token A comes *before* token B." The next notebook builds a task where order matters — and watches this model fail on it.

## Takeaways

- **`nn.Embedding`** turns discrete IDs into learnable vectors — the entry point for all of NLP.
- **Pooling** (mean/max/last) collapses a variable-length sequence to a fixed vector. Mean-pool = bag of words = order-independent.
- Match the model to the task: bag-of-words is perfect *when order doesn't matter*, useless when it does.

(Real datasets add **padding** to batch variable-length sequences and a mask so padding isn't pooled; we used fixed length `S` to stay focused.)